In [1]:
!pip install pandas scipy

import pandas as pd
from scipy.io import arff
import numpy as np
import os;

# Define the path to your ARFF file
file_path = os.path.join('/kaggle/input/credit-g/', 'dataset_31_credit-g.arff')

# 1. Load the ARFF data
df, meta = arff.loadarff(file_path)

# 2. Convert the structured array to a Pandas DataFrame
data = pd.DataFrame(df)

object_cols = data.select_dtypes(include=['object']).columns
for col in object_cols:
    # Use .apply() with a lambda function for robust decoding.
    # It checks if the value is a byte string (isinstance(x, bytes)) and decodes it using 'utf-8'.
    # Otherwise, it returns the value as is.
    data[col] = data[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

In [2]:
!pip install prince

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.0/179.0 kB 7.8 MB/s eta 0:00:00


In [3]:
import prince

X = data.drop('class', axis=1) # Assuming 'Class' is your target
y = data['class']

# Run FAMD with the max possible components (28)
famd_full = prince.FAMD(n_components=20, random_state=42)
famd_full = famd_full.fit(X)

# Get the eigenvalues (variance explained by each component)
eigenvalues = famd_full.eigenvalues_
# Get cumulative explained variance
cumulative_variance = eigenvalues.cumsum() / eigenvalues.sum()

# Print or plot to find the 'elbow' or threshold (e.g., 80% or 90%)
print(cumulative_variance)


[0.09996899 0.1708709  0.23379265 0.29388144 0.35076779 0.40325608
 0.45345861 0.50248777 0.55085749 0.59560017 0.6390412  0.68229787
 0.72503361 0.76734013 0.80861784 0.84869554 0.88804177 0.92562325
 0.96311174 1.        ]


In [4]:
famd = prince.FAMD(
    n_components=20,         # Number of dimensions to reduce to
    n_iter=3,                # Number of iterations
    random_state=42,         # for reproducibility
    engine="sklearn"         # utilize the sklearn engine
)

# Fit the model to your data
famd = famd.fit(data)


In [5]:
# Assuming 'data' is your original DataFrame
transformed_data = famd.transform(data)

# Display the first few rows of the new data
print(transformed_data.head())

# Check the shape of the new data: (number of original rows, 20 components)
print(transformed_data.shape)

component        0         1         2         3         4         5   \
0          0.569461 -5.430502 -3.338552 -0.457504 -0.013086 -2.118517   
1         -3.833185  2.984524  1.280651 -0.625003 -0.600535  3.575248   
2         -1.849568 -3.433232 -4.251738 -2.832679 -1.087824 -0.451026   
3          0.350177  2.772024 -4.365507 -1.124232 -4.097930 -3.431318   
4          6.528492  4.024065 -2.930878 -4.026111 -4.888955  0.514388   

component        6         7         8         9         10        11  \
0         -0.998781 -1.101743  0.857976  0.848579  0.802749  1.266692   
1         -0.929865 -3.589786  2.104552  0.280735 -0.035781 -0.815292   
2          3.118495  3.066895 -0.762249  2.395056 -1.049055 -3.337344   
3         -4.867407 -1.298780 -1.173850  4.316196  2.334720 -4.099640   
4          3.576001 -2.562433 -1.055981  1.854038  2.064038 -1.837404   

component        12        13        14        15        16        17  \
0         -1.487840  2.460801 -2.265169  0.908431

In [6]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
# X = sample.drop('Class', axis=1)
# y = sample['Class']
y = y.map({'bad': 1, 'good': 0})

# Perform a stratified train-test split to ensure both sets have the same fraud ratio (0.17%)
# Using 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    transformed_data,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [7]:
## Handle Imbalance (SMOTE Example for Training Data)

from imblearn.over_sampling import SMOTE

# Initialize SMOTE
smote = SMOTE(sampling_strategy='minority', random_state=42)

# Apply SMOTE ONLY to the training data
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Original Training Size: {len(y_train)}")
print(f"Original Training value counts: {y_train.value_counts()}\n")

print(f"SMOTE Training Size: {len(y_train_smote)}")
print(f"SMOTE Training value counts: {y_train_smote.value_counts()}\n")

print(f"SMOTE Training Fraud Rate: {y_train_smote.mean():.4f}")

Original Training Size: 700
Original Training value counts: class
0    490
1    210
Name: count, dtype: int64

SMOTE Training Size: 980
SMOTE Training value counts: class
1    490
0    490
Name: count, dtype: int64

SMOTE Training Fraud Rate: 0.5000


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler,StandardScaler

In [9]:
np.random.seed(0)
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ("classifier", LogisticRegression(random_state=0))
])

param_grid_lr_sm = param_grid_lr_sm = [
    {
        "classifier": [LogisticRegression(random_state=0)],
        "classifier__penalty": ['l1','l2'],
        "classifier__C": [0.1,0.01,0.001],
        "classifier__solver": ['liblinear'],
        "classifier__class_weight": [None],
        "classifier__max_iter": [1000,2000]
    }]

# Define Scorer (Good practice for imbalanced data like fraud detection)
# Using AUPRC (Average Precision Score) as suggested in the dataset context. It focuses only on the minority class (fraud).
# AUPRC gives a true picture of performance without being skewed by the large number of easy-to-classify non-fraud cases
scorer = make_scorer(average_precision_score)

grid_lr_sm = GridSearchCV(
    estimator=pipe_lr,
    scoring='average_precision',
    param_grid=param_grid_lr_sm,
    cv=5,
    n_jobs=-1
)

grid_lr_sm.fit(X_train_smote, y_train_smote)
print("Best score: {:.2f}".format(grid_lr_sm.best_score_))
print("Test set score: {:.2f}".format(grid_lr_sm.score(X_test, y_test)))
print("Best parameters: {}".format(grid_lr_sm.best_params_))

Best score: 0.97
Test set score: 0.93
Best parameters: {'classifier': LogisticRegression(random_state=0), 'classifier__C': 0.001, 'classifier__class_weight': None, 'classifier__max_iter': 1000, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}


In [10]:
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, classification_report
pred = grid_lr_sm.best_estimator_.predict(X_test)
proba = grid_lr_sm.best_estimator_.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, proba))
print("AUPRC:", average_precision_score(y_test, proba))
print(classification_report(y_test, pred))

Accuracy: 0.9066666666666666
ROC AUC: 0.9636507936507936
AUPRC: 0.9339267344201894
              precision    recall  f1-score   support

           0       0.94      0.92      0.93       210
           1       0.83      0.87      0.85        90

    accuracy                           0.91       300
   macro avg       0.89      0.90      0.89       300
weighted avg       0.91      0.91      0.91       300



In [11]:
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ("classifier",RandomForestClassifier())
    ])

search_space_rf_sm = [{
                 "classifier": [RandomForestClassifier()],
                 "classifier__n_estimators": [100, 300, 500],
                 "classifier__max_features": ['sqrt', 'log2'],
                 "classifier__max_depth": [10,20,30],
                 "classifier__min_samples_split": [2, 5, 10],
                 "classifier__criterion": ['entropy', 'log_loss'],
                 "classifier__n_jobs": [-1],
                 "classifier__class_weight": [None]
                }]
# Create grid search
grid_rf_sm = GridSearchCV(
    estimator=pipe_rf,
    param_grid=search_space_rf_sm,
    scoring='average_precision',
    cv=5,
    n_jobs=-1
)

# Fit grid search
grid_rf_sm.fit(X_train_smote, y_train_smote)

# Return all parameters and components of the pipeline as a dictionary
grid_rf_sm.best_estimator_.get_params()

# View best model
grid_rf_sm.best_estimator_.get_params()["classifier"]

# Predict target vector
grid_rf_sm.predict(X_test)

print("Best accuracy: {:.2f}".format(grid_rf_sm.best_score_))
print("Test set score: {:.2f}".format(grid_rf_sm.score(X_test, y_test)))
print("Best parameters: {}".format(grid_rf_sm.best_params_))

results_rf = grid_rf_sm.cv_results_

Best accuracy: 0.98
Test set score: 0.91
Best parameters: {'classifier': RandomForestClassifier(), 'classifier__class_weight': None, 'classifier__criterion': 'entropy', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_split': 2, 'classifier__n_estimators': 500, 'classifier__n_jobs': -1}


In [12]:
pred = grid_rf_sm.best_estimator_.predict(X_test)
proba = grid_rf_sm.best_estimator_.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, proba))
print("AUPRC:", average_precision_score(y_test, proba))
print(classification_report(y_test, pred))

Accuracy: 0.8933333333333333
ROC AUC: 0.9513756613756613
AUPRC: 0.9097233292264586
              precision    recall  f1-score   support

           0       0.93      0.91      0.92       210
           1       0.81      0.84      0.83        90

    accuracy                           0.89       300
   macro avg       0.87      0.88      0.87       300
weighted avg       0.89      0.89      0.89       300



In [13]:
!pip install xgboost

In [14]:
from collections import Counter

counter = Counter(y_train)
scale_pos_weight = counter[0] / counter[1]

pipe_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ("classifier",xgb.XGBClassifier())])

# Create dictionary with candidate learning algorithms and their hyperparameters
search_space_xgb = [
               {"classifier": [xgb.XGBClassifier()],
                 "classifier__n_estimators": [10,100,1000],
                 "classifier__max_depth": [3,6,10],
                 "classifier__learning_rate": [0.01, 0.05, 0.1],
                 "classifier__n_jobs": [-1],
                 "classifier__gamma": [0.5,0,1],
                 "classifier__scale_pos_weight": [scale_pos_weight]
                }]

# Create grid search
grid_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=search_space_xgb,
    scoring='average_precision',
    cv=5,
    n_jobs=-1
)
# Fit grid search
grid_xgb.fit(X_train, y_train)

# Return all parameters and components of the pipeline as a dictionary
grid_xgb.best_estimator_.get_params()

# View best model
grid_xgb.best_estimator_.get_params()["classifier"]

# Predict target vector
grid_xgb.predict(X_test)

print("Best accuracy: {:.2f}".format(grid_xgb.best_score_))
print("Test set score: {:.2f}".format(grid_xgb.score(X_test, y_test)))
print("Best parameters: {}".format(grid_xgb.best_params_))

results_xgb = grid_xgb.cv_results_

Best accuracy: 0.92
Test set score: 0.94
Best parameters: {'classifier': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...), 'classifier__gamma': 0, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 3, 'classifier__n_estimators': 1000, 'classifier__n_jobs': -1, 'classifier__scale_

In [16]:
pred = grid_xgb.best_estimator_.predict(X_test)
proba = grid_xgb.best_estimator_.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, proba))
print("AUPRC:", average_precision_score(y_test, proba))
print(classification_report(y_test, pred))

Accuracy: 0.9233333333333333
ROC AUC: 0.9695767195767196
AUPRC: 0.9395821322174374
              precision    recall  f1-score   support

           0       0.94      0.95      0.95       210
           1       0.88      0.87      0.87        90

    accuracy                           0.92       300
   macro avg       0.91      0.91      0.91       300
weighted avg       0.92      0.92      0.92       300

